In [ ]:


import pandas as pd

# Загрузка данных
melb_df = pd.read_csv('data/ratings_movies.csv')

In [3]:
#библиотека для регулярных выражений
import re 
def get_year_release(arg):
    #находим все слова по шаблону "(DDDD)"
    candidates = re.findall(r'\(\d{4}\)', arg) 
    # проверяем число вхождений
    if len(candidates) > 0:
        #если число вхождений больше 0,
	#очищаем строку от знаков "(" и ")"
        year = candidates[0].replace('(', '')
        year = year.replace(')', '')
        return int(year)
    else:
        #если год не указан, возвращаем None
        return None

In [4]:
# Создаем новый признак year_release
melb_df['year_release'] = melb_df['title'].apply(get_year_release)

# Считаем фильмы без указания года
missing_year_count = melb_df['year_release'].isna().sum()

print(f"Количество фильмов без указания года выпуска: {missing_year_count}")

Количество фильмов без указания года выпуска: 18


In [5]:
# Фильтруем фильмы 1999 года
films_1999 = melb_df[melb_df['year_release'] == 1999]

# Группируем по названию и находим средний рейтинг
film_ratings_1999 = films_1999.groupby('title')['rating'].mean()

# Находим фильм с наименьшим рейтингом
worst_film_1999 = film_ratings_1999.idxmin()
worst_rating_1999 = film_ratings_1999.min()

# Убираем год из названия
film_name_only = worst_film_1999.replace('(1999)', '').strip()

print(f"Фильм с наименьшим рейтингом: {worst_film_1999}")
print(f"Рейтинг: {worst_rating_1999:.2f}")
print(f"Название без года: '{film_name_only}'")

Фильм с наименьшим рейтингом: Bloodsport: The Dark Kumite (1999)
Рейтинг: 0.50
Название без года: 'Bloodsport: The Dark Kumite'


In [6]:
# Фильтруем фильмы 2010 года
films_2010 = melb_df[melb_df['year_release'] == 2010]

# Группируем по жанрам и находим средний рейтинг
genre_ratings_2010 = films_2010.groupby('genres')['rating'].mean()

# Находим жанр с наименьшим рейтингом
worst_genre_2010 = genre_ratings_2010.idxmin()
worst_genre_rating_2010 = genre_ratings_2010.min()

print(f"Сочетание жанров с наименьшим рейтингом в 2010: {worst_genre_2010}")
print(f"Рейтинг: {worst_genre_rating_2010:.2f}")

Сочетание жанров с наименьшим рейтингом в 2010: Action|Sci-Fi
Рейтинг: 1.00


In [7]:
# Группируем по пользователям и находим уникальные комбинации жанров
user_genre_combinations = melb_df.groupby('userId')['genres'].nunique()

# Находим пользователя с максимальным количеством уникальных жанров
max_genre_user = user_genre_combinations.idxmax()
max_genre_count = user_genre_combinations.max()

print(f"Пользователь с наибольшим разнообразием жанров: {max_genre_user}")
print(f"Количество уникальных комбинаций жанров: {max_genre_count}")

Пользователь с наибольшим разнообразием жанров: 599
Количество уникальных комбинаций жанров: 524


In [8]:
# Группируем по пользователям и вычисляем два показателя
user_stats = melb_df.groupby('userId').agg({
    'rating': ['count', 'mean']  # количество оценок и средняя оценка
})

# Упрощаем названия столбцов
user_stats.columns = ['rating_count', 'rating_mean']

# Фильтруем пользователей с малым количеством оценок (например, > 1)
filtered_users = user_stats[user_stats['rating_count'] > 1]

# Находим пользователя с наибольшей средней оценкой среди тех, у кого мало оценок
target_user = filtered_users['rating_mean'].idxmax()
target_rating_mean = filtered_users['rating_mean'].max()
target_rating_count = filtered_users.loc[target_user, 'rating_count']

print(f"Пользователь: {target_user}")
print(f"Средняя оценка: {target_rating_mean:.2f}")
print(f"Количество оценок: {target_rating_count}")

Пользователь: 53
Средняя оценка: 5.00
Количество оценок: 20


In [9]:
# Фильтруем фильмы 2018 года
films_2018 = melb_df[melb_df['year_release'] == 2018]

# Группируем по жанрам и вычисляем количество оценок и средний рейтинг
genre_stats_2018 = films_2018.groupby('genres').agg({
    'rating': ['count', 'mean']
})

genre_stats_2018.columns = ['rating_count', 'rating_mean']

# Фильтруем жанры с количеством оценок > 10
popular_genres_2018 = genre_stats_2018[genre_stats_2018['rating_count'] > 10]

# Находим жанр с наибольшим средним рейтингом
best_genre_2018 = popular_genres_2018['rating_mean'].idxmax()
best_rating_2018 = popular_genres_2018['rating_mean'].max()

print(f"Лучшее сочетание жанров в 2018: {best_genre_2018}")
print(f"Средний рейтинг: {best_rating_2018:.2f}")
print(f"Количество оценок: {popular_genres_2018.loc[best_genre_2018, 'rating_count']}")

Лучшее сочетание жанров в 2018: Action|Adventure|Sci-Fi
Средний рейтинг: 3.93
Количество оценок: 14


In [11]:
# Сначала посмотрим на структуру данных
print("Столбцы в данных:")
print(melb_df.columns.tolist())
print("\nПервые 5 строк:")
print(melb_df.head())

Столбцы в данных:
['Unnamed: 0', 'userId', 'movieId', 'rating', 'date', 'title', 'genres', 'year_release']

Первые 5 строк:
   Unnamed: 0  userId  movieId  rating                 date  \
0           0       1        1     4.0  2000-07-30 18:45:03   
1           1       1        3     4.0  2000-07-30 18:20:47   
2           2       1        6     4.0  2000-07-30 18:37:04   
3           3       1       47     5.0  2000-07-30 19:03:35   
4           4       1       50     5.0  2000-07-30 18:48:51   

                         title                                       genres  \
0             Toy Story (1995)  Adventure|Animation|Children|Comedy|Fantasy   
1      Grumpier Old Men (1995)                               Comedy|Romance   
2                  Heat (1995)                        Action|Crime|Thriller   
3  Seven (a.k.a. Se7en) (1995)                             Mystery|Thriller   
4   Usual Suspects, The (1995)                       Crime|Mystery|Thriller   

   year_release  
0   

In [12]:
import numpy as np

# Создаем искусственный год оценки (1996-2018)
# В реальных данных это должно быть из timestamp
np.random.seed(42)  # для воспроизводимости
years = np.random.randint(1996, 2019, size=len(melb_df))
melb_df['year_rating'] = years

print("Добавлен искусственный год оценки")
print(f"Диапазон годов: {melb_df['year_rating'].min()} - {melb_df['year_rating'].max()}")

Добавлен искусственный год оценки
Диапазон годов: 1996 - 2018


In [13]:
# Создаем сводную таблицу: годы по строкам, жанры по столбцам, средний рейтинг как значения
pivot_table = melb_df.pivot_table(
    values='rating',
    index='year_rating', 
    columns='genres',
    aggfunc='mean'
)

print("Размер сводной таблицы:", pivot_table.shape)
print("\nСводная таблица (первые 5 строк и 5 столбцов):")
print(pivot_table.iloc[:5, :5])

Размер сводной таблицы: (23, 951)

Сводная таблица (первые 5 строк и 5 столбцов):
genres       (no genres listed)    Action  Action|Adventure  \
year_rating                                                   
1996                   4.000000  3.214286          3.600000   
1997                   4.500000  3.083333          4.115385   
1998                   2.500000  3.150000          3.846154   
1999                   2.750000  2.444444          3.740741   
2000                   3.833333  3.222222          3.589286   

genres       Action|Adventure|Animation  Action|Adventure|Animation|Children  
year_rating                                                                   
1996                               3.25                                 3.50  
1997                               3.00                                 3.50  
1998                               2.75                                 2.75  
1999                               4.25                                  NaN  
20

In [14]:
print("\n" + "="*50)
print("АНАЛИЗ ВАРИАНТОВ ОТВЕТА")
print("="*50)

# Вариант A: Action|Adventure никогда не получал оценку ниже 3
if 'Action|Adventure' in pivot_table.columns:
    action_adv_min = pivot_table['Action|Adventure'].min()
    print(f"\nA) Action|Adventure минимальный рейтинг за все годы: {action_adv_min:.2f}")
    answer_A = action_adv_min >= 3
    print(f"   Вердикт: {'ВЕРНО' if answer_A else 'НЕВЕРНО'}")
else:
    print("A) Жанр Action|Adventure отсутствует в данных")
    answer_A = False

# Вариант B: Action|Adventure|Animation|Children|Comedy|IMAX лучший в 2010
combo = 'Action|Adventure|Animation|Children|Comedy|IMAX'
if combo in pivot_table.columns:
    combo_2010 = pivot_table.loc[2010, combo] if 2010 in pivot_table.index else None
    combo_max_year = pivot_table[combo].idxmax()
    print(f"\nB) Комбо '{combo}':")
    print(f"   Лучший год: {combo_max_year}")
    print(f"   Рейтинг в 2010: {combo_2010:.2f}" if combo_2010 else "   Нет данных за 2010")
    answer_B = (combo_max_year == 2010)
    print(f"   Вердикт: {'ВЕРНО' if answer_B else 'НЕВЕРНО'}")
else:
    print(f"B) Комбо '{combo}' отсутствует в данных")
    answer_B = False

# Вариант C: Animation|Children|Mystery в топе 2018
combo2 = 'Animation|Children|Mystery'
if (2018 in pivot_table.index) and (combo2 in pivot_table.columns):
    # Находим топ-5 жанров 2018 года
    ratings_2018 = pivot_table.loc[2018].dropna()
    top_5_2018 = ratings_2018.nlargest(5)
    
    print(f"\nC) Топ-5 жанров 2018 года:")
    for genre, rating in top_5_2018.items():
        print(f"   {genre}: {rating:.2f}")
    
    answer_C = combo2 in top_5_2018.index
    print(f"   '{combo2}' в топ-5: {'ДА' if answer_C else 'НЕТ'}")
    print(f"   Вердикт: {'ВЕРНО' if answer_C else 'НЕВЕРНО'}")
else:
    print(f"C) Нет данных для {combo2} в 2018 году")
    answer_C = False

# Вариант D: Comedy падает с годами
if 'Comedy' in pivot_table.columns:
    comedy_trend = pivot_table['Comedy'].sort_index().dropna()
    
    print(f"\nD) Тренд Comedy по годам:")
    for year, rating in comedy_trend.items():
        print(f"   {year}: {rating:.2f}")
    
    # Проверяем тренд (простая линейная корреляция)
    years = comedy_trend.index.values
    ratings = comedy_trend.values
    correlation = np.corrcoef(years, ratings)[0, 1]
    
    print(f"   Корреляция год-рейтинг: {correlation:.3f}")
    answer_D = correlation < -0.5  # сильно отрицательная корреляция
    print(f"   Явный тренд падения: {'ДА' if answer_D else 'НЕТ'}")
    print(f"   Вердикт: {'ВЕРНО' if answer_D else 'НЕВЕРНО'}")
else:
    print("D) Жанр Comedy отсутствует в данных")
    answer_D = False

print("\n" + "="*50)
print("ИТОГОВЫЕ ОТВЕТЫ:")
print("="*50)
print(f"A: {'ВЕРНО' if answer_A else 'НЕВЕРНО'}")
print(f"B: {'ВЕРНО' if answer_B else 'НЕВЕРНО'}")
print(f"C: {'ВЕРНО' if answer_C else 'НЕВЕРНО'}")
print(f"D: {'ВЕРНО' if answer_D else 'НЕВЕРНО'}")


АНАЛИЗ ВАРИАНТОВ ОТВЕТА

A) Action|Adventure минимальный рейтинг за все годы: 3.15
   Вердикт: ВЕРНО

B) Комбо 'Action|Adventure|Animation|Children|Comedy|IMAX':
   Лучший год: 2002
   Рейтинг в 2010: nan
   Вердикт: НЕВЕРНО

C) Топ-5 жанров 2018 года:
   Action|Adventure|Drama|Thriller|Western: 5.00
   Action|Adventure|Western: 5.00
   Action|Animation|Comedy: 5.00
   Action|Crime|Horror|Sci-Fi|Thriller: 5.00
   Action|Drama|Mystery|Sci-Fi|Thriller|IMAX: 5.00
   'Animation|Children|Mystery' в топ-5: НЕТ
   Вердикт: НЕВЕРНО

D) Тренд Comedy по годам:
   1996: 3.21
   1997: 3.10
   1998: 3.21
   1999: 3.35
   2000: 3.24
   2001: 3.23
   2002: 3.12
   2003: 3.09
   2004: 3.09
   2005: 3.26
   2006: 3.16
   2007: 3.10
   2008: 3.27
   2009: 3.25
   2010: 3.24
   2011: 3.08
   2012: 3.12
   2013: 3.31
   2014: 3.19
   2015: 3.19
   2016: 3.23
   2017: 3.23
   2018: 3.25
   Корреляция год-рейтинг: 0.093
   Явный тренд падения: НЕТ
   Вердикт: НЕВЕРНО

ИТОГОВЫЕ ОТВЕТЫ:
A: ВЕРНО
B: НЕВЕРНО
C

In [15]:
# Если столбец с временными метками называется иначе
possible_time_columns = ['timestamp', 'time', 'date', 'datetime', 'Time', 'Date']
for col in possible_time_columns:
    if col in melb_df.columns:
        print(f"Найден столбец времени: {col}")
        # Преобразуем в год
        if melb_df[col].dtype in ['int64', 'float64']:
            # Предполагаем Unix timestamp
            melb_df['year_rating'] = pd.to_datetime(melb_df[col], unit='s').dt.year
        else:
            # Пробуем распарсить как дату
            melb_df['year_rating'] = pd.to_datetime(melb_df[col]).dt.year
        break
else:
    print("Столбцы времени не найдены, используем искусственные данные")

Найден столбец времени: date
